# Dataset Audit

This notebook audits the "Sternberg Difficult" EEG dataset under `data/raw/` and answers the main project questions before any modeling begins.

The goal is not to change the raw data. Instead, we read it carefully, check that the organization is valid, and extract the metadata we need for the analysis pipeline.

## Why this audit matters

Before fitting a latent-state model, we need to know:
- how many participants exist
- how the EEG data is structured
- how trials and events are encoded
- where behavioral variables are stored
- whether electrode coordinates are available

This helps with both scientific validity and reproducibility.

In [3]:
from pathlib import Path
import json
import pandas as pd

# The notebook should work whether it is run from the project root or from the notebooks/ folder.
project_root = Path.cwd()
if not (project_root / 'data' / 'raw').exists():
    project_root = project_root.parent

DATA_ROOT = project_root / 'data' / 'raw'
print(f'Project root: {project_root}')
print(f'Dataset root: {DATA_ROOT}')
print(f'Dataset exists: {DATA_ROOT.exists()}')

Project root: d:\Research\eeg-neural-state-dynamics
Dataset root: d:\Research\eeg-neural-state-dynamics\data\raw
Dataset exists: True


In [4]:
participants_path = DATA_ROOT / 'participants.tsv'
participants_df = pd.read_csv(participants_path, sep='\t')

print(f'Number of subjects: {len(participants_df)}')
print('Subject IDs:')
print(participants_df['participant_id'].tolist())

participants_df.head()

Number of subjects: 48
Subject IDs:
['sub-04', 'sub-05', 'sub-06', 'sub-07', 'sub-08', 'sub-09', 'sub-10', 'sub-11', 'sub-12', 'sub-13', 'sub-14', 'sub-15', 'sub-16', 'sub-17', 'sub-18', 'sub-19', 'sub-21', 'sub-22', 'sub-23', 'sub-24', 'sub-25', 'sub-26', 'sub-27', 'sub-28', 'sub-29', 'sub-30', 'sub-31', 'sub-32', 'sub-33', 'sub-34', 'sub-35', 'sub-36', 'sub-37', 'sub-38', 'sub-39', 'sub-40', 'sub-41', 'sub-42', 'sub-43', 'sub-44', 'sub-45', 'sub-46', 'sub-47', 'sub-48', 'sub-49', 'sub-50', 'sub-51', 'sub-52']


,participant_id,age,sex,accuracy_3,accuracy_6,accuracy_9,accuracy_12,accuracy_15,MT,NfC,PA,Raven,SE,hardiness,hardComm,hardContr,hardChall,AM
0,sub-04,23,f,1.000,0.900,0.725,0.775,0.621622,41,80,31,53,33,75,30,23,19,15
1,sub-05,20,f,1.000,0.875,0.850,0.575,0.550000,29,95,52,49,27,70,31,19,17,18
2,sub-06,21,m,1.000,0.975,0.675,0.700,0.675000,23,98,47,51,22,53,24,12,14,13
3,sub-07,21,m,1.000,0.925,0.825,0.825,0.775000,23,82,44,53,25,65,26,18,18,13
4,sub-08,19,f,0.975,0.850,0.725,0.575,0.525000,30,96,47,55,33,70,30,20,17,15


## EEG recording metadata

Now we check the EEG recording metadata. The most important fields are:
- number of channels
- sampling frequency
- recording duration

These values are stored in the sidecar JSON file for each subject. All downstream preprocessing, time-windowing, and model input dimensions depend on these values.

In [8]:
subject_dirs = sorted(DATA_ROOT.glob('sub-*'))
eeg_summary_rows = []

for subject_dir in subject_dirs:
    eeg_dir = subject_dir / 'ses-01' / 'eeg'
    eeg_jsons = sorted(eeg_dir.glob('*_eeg.json'))
    if not eeg_jsons:
        continue

    eeg_json_path = eeg_jsons[0]
    with open(eeg_json_path, 'r', encoding='utf-8') as f:
        meta = json.load(f)

    eeg_summary_rows.append({
        'subject_id': subject_dir.name,
        'channels': meta.get('EEGChannelCount'),
        'sampling_frequency_hz': meta.get('SamplingFrequency'),
        'recording_duration_s': meta.get('RecordingDuration'),
        'task_name': meta.get('TaskName'),
        'reference': meta.get('EEGReference'),
    })

eeg_summary = pd.DataFrame(eeg_summary_rows)
print(eeg_summary.head())
print(f'\nUnique channel counts: {sorted(eeg_summary["channels"].unique().tolist())}')
print(f'Unique sampling frequencies: {sorted(eeg_summary["sampling_frequency_hz"].unique().tolist())}')
print(f'Min duration: {eeg_summary["recording_duration_s"].min()} s')
print(f'Max duration: {eeg_summary["recording_duration_s"].max()} s')
print(f'Mean duration: {eeg_summary["recording_duration_s"].mean()} s')

  subject_id  channels  sampling_frequency_hz  recording_duration_s  \
0     sub-04        63                 1000.0              1196.549   
1     sub-05        63                 1000.0              1156.849   
2     sub-06        63                 1000.0              1244.899   
3     sub-07        63                 1000.0              1121.749   
4     sub-08        63                 1000.0              1248.799   

   task_name reference  
0  STERNBERG       FCz  
1  STERNBERG       FCz  
2  STERNBERG       FCz  
3  STERNBERG       FCz  
4  STERNBERG       FCz  

Unique channel counts: [63]
Unique sampling frequencies: [1000.0]
Min duration: 1100.949 s
Max duration: 1753.949 s
Mean duration: 1267.5854583333332 s


## Event structure and trial encoding

Events encode trial timing and experimental markers. The JSON sidecar tells us what the event names mean.

The key questions here are:
- How are trials encoded?
- Where are the relevant phases marked?

In [9]:
example_subject = subject_dirs[0]
example_eeg_dir = example_subject / 'ses-01' / 'eeg'
example_events_json = sorted(example_eeg_dir.glob('*_events.json'))[0]
example_events_tsv = sorted(example_eeg_dir.glob('*_events.tsv'))[0]

with open(example_events_json, 'r', encoding='utf-8') as f:
    events_json = json.load(f)

print('Events JSON structure:')
print(events_json.keys())
print('\ntrial_type levels:')
for k, v in events_json['trial_type']['Levels'].items():
    print(f'  {k}: {v}')

events_df = pd.read_csv(example_events_tsv, sep='\t')
print('\nFirst rows of events.tsv:')
print(events_df.head(10).to_string(index=False))

Events JSON structure:
dict_keys(['onset', 'duration', 'trial_type', 'value', 'sample'])

trial_type levels:
  New Segment/: Start of the recording
  Stimulus/S 10: Inorrect response
  Stimulus/S 11: Correct response
  Stimulus/S 31: 3-letter sample; test object absent in the encoding set
  Stimulus/S 32: 3-letter sample; test object present in the encoding set
  Stimulus/S 61: 6-letter sample; test object absent in the encoding set
  Stimulus/S 62: 6-letter sample; test object present in the encoding set
  Stimulus/S 91: 9-letter sample; test object absent in the encoding set
  Stimulus/S 92: 9-letter sample; test object present in the encoding set
  Stimulus/S121: 12-letter sample; test object absent in the encoding set
  Stimulus/S122: 12-letter sample; test object present in the encoding set
  Stimulus/S151: 15-letter sample; test object absent in the encoding set
  Stimulus/S152: 15-letter sample; test object present in the encoding set

First rows of events.tsv:
 onset  duration 

### Interpreting the event codes

The sidecar JSON reveals how the codes are meant to be understood. For example:
- `Stimulus/S 31` = 3-letter sample; item absent
- `Stimulus/S 32` = 3-letter sample; item present
- `Stimulus/S 61` = 6-letter sample; item absent
- `Stimulus/S 62` = 6-letter sample; item present
- and so on up to 15-letter trials

This tells us that the dataset encodes trial identity via the stimulus code, and the load size is embedded in the code itself.

Each event marker is duplicated with the same data for every column except the duration, which does not add any meaningful value. Therefore, each entry needs to be deduplicated.

The dataset does not explicitly mark the beginning of retention and retrieval with separate trigger labels, which is why the dataset README explains that these phase boundaries must be inferred from the task timing. It states that the task includes:
- encoding: 1500 ms
- retention: 2000 ms
- retrieval: 1500 ms

In [1]:
# Task timing as described in the dataset README.
encoding_ms = 1500
retention_ms = 2000
retrieval_ms = 1500

# A stimulus event marks the start of the trial.
# Then:
#   0 to 1500 ms   -> encoding
#   1500 to 3500 ms -> retention
#   3500 ms onward -> retrieval

def phase_from_onset_ms(onset_ms):
    if 0 <= onset_ms < encoding_ms:
        return "encoding"
    if encoding_ms <= onset_ms < encoding_ms + retention_ms:
        return "retention"
    if encoding_ms + retention_ms <= onset_ms < encoding_ms + retention_ms + retrieval_ms:
        return "retrieval"
    return "outside_trial"

sample_onsets = [0, 500, 1500, 2000, 3500, 5000]
for t in sample_onsets:
    print(f'{t} ms -> {phase_from_onset_ms(t)}')

0 ms -> encoding
500 ms -> encoding
1500 ms -> retention
2000 ms -> retention
3500 ms -> retrieval
5000 ms -> outside_trial


## Behavioral data

The project is interested in cognitive load, accuracy, and reaction time. The dataset stores some of these in the participant table, and some in the event files.

We are interested in:
- Where is `load`?
- Where is `accuracy`?
- Where is `reaction time`?

In [12]:
# Behavioral information at the participant level
print('Participant-level columns:')
print(participants_df.columns.tolist())
print('\nAccuracy columns:')
print([col for col in participants_df.columns if 'accuracy' in col])
print('\nLoad-related information is encoded in stimulus codes, not in a separate load column.')

# Example: a few rows of the participant table
participants_df[['participant_id', 'accuracy_3', 'accuracy_6', 'accuracy_9', 'accuracy_12', 'accuracy_15']].head()

Participant-level columns:
['participant_id', 'age', 'sex', 'accuracy_3', 'accuracy_6', 'accuracy_9', 'accuracy_12', 'accuracy_15', 'MT', 'NfC', 'PA', 'Raven', 'SE', 'hardiness', 'hardComm', 'hardContr', 'hardChall', 'AM']

Accuracy columns:
['accuracy_3', 'accuracy_6', 'accuracy_9', 'accuracy_12', 'accuracy_15']

Load-related information is encoded in stimulus codes, not in a separate load column.


,participant_id,accuracy_3,accuracy_6,accuracy_9,accuracy_12,accuracy_15
0,sub-04,1.000,0.900,0.725,0.775,0.621622
1,sub-05,1.000,0.875,0.850,0.575,0.550000
2,sub-06,1.000,0.975,0.675,0.700,0.675000
3,sub-07,1.000,0.925,0.825,0.825,0.775000
4,sub-08,0.975,0.850,0.725,0.575,0.525000


In [ ]:
# Trial-level behavioral interpretation from event codes
# 10 = incorrect response
# 11 = correct response
event_value_counts = events_df['value'].value_counts().head(10)
print(event_value_counts.to_string())

# Reaction time is not stored as a dedicated column in the table.
# response_time = response_event_onset - stimulus_onset
# This requires linking each response event to the preceding trial stimulus.
print('\nReaction time is inferred from timing differences between stimulus events and response events.')

value
11     318
10      76
122     40
121     40
62      40
61      40
91      40
92      40
31      40
32      40

Reaction time is inferred from timing differences between stimulus events and response events.


### Interpretation

- **Load** is represented by the stimulus code family, such as `31/32`, `61/62`, `91/92`, `121/122`, and `151/152`.
- **Accuracy** is stored in the participant table as `accuracy_3`, `accuracy_6`, `accuracy_9`, `accuracy_12`, and `accuracy_15`.
- **Reaction time** is not a direct column in `participants.tsv`; it must be computed from event timing by subtracting the onset of the stimulus from the onset of the response.

## Electrode metadata and coordinate system

The final part of the audit checks whether electrode coordinates are available, and what coordinate system they use.

This matters because sensor-space EEG interpretations should be expressed carefully, and electrode coordinates are also needed for plotting and source-space work later.

In [14]:
coordsystem_path = example_eeg_dir / f'{example_subject.name}_ses-01_space-CapTrak_coordsystem.json'
with open(coordsystem_path, 'r', encoding='utf-8') as f:
    coordsystem = json.load(f)

print('Coordinate system metadata:')
print(json.dumps({
    'EEGCoordinateSystem': coordsystem.get('EEGCoordinateSystem'),
    'EEGCoordinateUnits': coordsystem.get('EEGCoordinateUnits'),
    'AnatomicalLandmarkCoordinateSystem': coordsystem.get('AnatomicalLandmarkCoordinateSystem'),
    'AnatomicalLandmarkCoordinateUnits': coordsystem.get('AnatomicalLandmarkCoordinateUnits'),
}, indent=2))

electrodes_path = example_eeg_dir / f'{example_subject.name}_ses-01_space-CapTrak_electrodes.tsv'
electrodes_df = pd.read_csv(electrodes_path, sep='\t')
print('\nFirst rows of electrode coordinates:')
print(electrodes_df.head(10).to_string(index=False))

Coordinate system metadata:
{
  "EEGCoordinateSystem": "CapTrak",
  "EEGCoordinateUnits": "m",
  "AnatomicalLandmarkCoordinateSystem": "CapTrak",
  "AnatomicalLandmarkCoordinateUnits": "m"
}

First rows of electrode coordinates:
name             x         y             z  impedance
 Fp1 -2.935661e-02  0.090350  5.817072e-18        NaN
  Fz  4.113291e-18  0.067175  6.717514e-02        NaN
  F3 -5.177571e-02  0.063938  4.750000e-02        NaN
  F7 -7.685661e-02  0.055840  5.817072e-18        NaN
 FT9 -8.316795e-02  0.027023 -3.711946e-02        NaN
 FC5 -8.279938e-02  0.031784  3.404496e-02        NaN
 FC1 -3.398867e-02  0.035196  8.143089e-02        NaN
  C3 -6.717514e-02  0.000000  6.717514e-02        NaN
  T7 -9.500000e-02  0.000000  5.817072e-18        NaN
 TP9 -8.316795e-02 -0.027023 -3.711946e-02        NaN


### What the coordinate metadata tells us

The dataset provides electrode coordinates in a coordinate system called `CapTrak`. The sidecar notes that it follows a typical RAS-style orientation with landmarks such as:
- NAS
- LPA
- RPA

This means the electrode coordinates are available and can be used for sensor-space plotting or later source analysis.

However, we will not be overinterpreting scalp patterns as cortical networks unless additional evidence exists. This audit is useful merely because it shows what we know and what we may conclude from the dataset.

## Final audit summary

- Subjects: check number and IDs from `participants.tsv`
- EEG recording: 63 channels, 1000 Hz, recording duration varies by participant
- Trial encoding: stimulus codes encode load and presence/absence conditions
- Task phases: explicit markers are not present; encoding, retention, and retrieval are inferred from timing rules
- Behavioral variables: participant-level accuracy in `participants.tsv`, load encoded in event codes, reaction time inferred from timing
- Electrode metadata: coordinates available, coordinate system = `CapTrak`